# M4U3 | Concrete Crack Detection
**Author:** Ahmed El Basyouni

Academic YOLOv8n object detection for visible concrete cracks. Human review is
required; detections do not establish structural safety or severity.

**Before Run all:** upload the provided best.pt to GitHub Release v1.0 and publish
this repository on main. The notebook uses public URLs and contains no manual
uploads, credential prompts or Roboflow API keys. A successful fresh Colab test
is still required. Default mode loads the completed 30-epoch weights.


## Install libraries

In [ ]:
%pip install -q ultralytics==8.3.40 PyYAML==6.0.2


## Environment and download utilities

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, urllib.request, zipfile, json, time, random
import yaml, torch, ultralytics, pandas as pd
from ultralytics import YOLO
from IPython.display import display, Image
STARTED_AT = datetime.now(timezone.utc).isoformat()
START_TIME = time.perf_counter()
print('Ultralytics:', ultralytics.__version__)
print('PyTorch:', torch.__version__)
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
DEVICE = 0 if torch.cuda.is_available() else 'cpu'
RESULTS = Path('/content/results'); RESULTS.mkdir(exist_ok=True)
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as stream:
        for block in iter(lambda: stream.read(1 << 20), b''):
            h.update(block)
    return h.hexdigest()
def download_verified(url, path, expected_hash):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    if not path.exists() or sha256_file(path) != expected_hash:
        urllib.request.urlretrieve(url, path)
    actual = sha256_file(path)
    if actual != expected_hash:
        path.unlink(missing_ok=True)
        raise ValueError(f'Checksum mismatch: {actual}')
    return path


## Frozen dataset and annotation conversion
405 images: 324 train / 81 validation. Auto-orient, stretch 640x640; no Roboflow augmentation. One polygon is converted to its enclosing box.

In [ ]:
DATA_URL = 'https://github.com/aalbassiony-png/concrete-crack-detection/releases/download/v1.0/concrete-cracks-v1-yolo11.zip'
DATA_SHA256 = '4f6cf7fb5f8caefd2621fc801a8f337773b08c6c578a10534d222794a4c0faba'
ZIP = download_verified(DATA_URL, '/content/dataset.zip', DATA_SHA256)
ROOT = Path('/content/dataset'); ROOT.mkdir(exist_ok=True)
with zipfile.ZipFile(ZIP) as archive:
    for name in archive.namelist():
        target = (ROOT / name).resolve()
        if not target.is_relative_to(ROOT.resolve()):
            raise ValueError('Unsafe archive member')
    archive.extractall(ROOT)
hits = sorted(ROOT.rglob('data.yaml'))
if len(hits) != 1:
    raise ValueError(f'Expected one data.yaml, found {len(hits)}')
CFG_PATH = hits[0]; DATASET_ROOT = CFG_PATH.parent
cfg = yaml.safe_load(CFG_PATH.read_text())
cfg.update(path=str(DATASET_ROOT.resolve()), train='train/images', val='valid/images')
cfg.pop('test', None)
CFG_PATH.write_text(yaml.safe_dump(cfg, sort_keys=False))
converted = 0
for split, expected in [('train', 324), ('valid', 81)]:
    images = [p for p in (DATASET_ROOT / split / 'images').iterdir() if p.suffix.lower() in {'.jpg','.jpeg','.png','.webp'}]
    assert len(images) == expected, (split, len(images))
    for image in images:
        label = DATASET_ROOT / split / 'labels' / (image.stem + '.txt')
        assert label.exists(), label
        rows = []
        for line in label.read_text().splitlines():
            if not line.strip(): continue
            values = list(map(float, line.split()))
            assert values[0] == 0
            if len(values) > 5:
                assert (len(values)-1) % 2 == 0
                xs, ys = values[1::2], values[2::2]
                values = [0,(min(xs)+max(xs))/2,(min(ys)+max(ys))/2,max(xs)-min(xs),max(ys)-min(ys)]
                converted += 1
            assert len(values) == 5 and all(0 <= v <= 1 for v in values[1:])
            rows.append(' '.join(str(v) for v in values))
        label.write_text('\n'.join(rows) + ('\n' if rows else ''))
    print(split, len(images), 'images')
print('Polygon annotations converted:', converted)


## Verified trained weights

In [ ]:
WEIGHTS_URL = 'https://github.com/aalbassiony-png/concrete-crack-detection/releases/download/v1.0/best.pt'
WEIGHTS_SHA256 = 'ef01306429e31ed259def70a3841ab2d54c58254e4f665b17e32c7f2d085f4fd'
BEST_WEIGHTS = download_verified(WEIGHTS_URL, '/content/weights/best.pt', WEIGHTS_SHA256)
model = YOLO(str(BEST_WEIGHTS))
print('Verified trained weights:', WEIGHTS_SHA256)


## Optional full training
Default: saved-weight verification and inference. Enabling full training creates a separate experiment; metrics may vary.

In [ ]:
# Default reproduces evaluation by loading the completed 30-epoch weights.
# Set True only to run a new 30-epoch training experiment.
RUN_FULL_TRAINING = False
if RUN_FULL_TRAINING:
    trainer_model = YOLO('yolov8n.pt')
    trainer_model.train(data=str(CFG_PATH), epochs=30, batch=16, imgsz=640,
                        seed=42, device=DEVICE, workers=2, patience=0,
                        project=str(RESULTS), name='retrained_yolov8n_cracks',
                        exist_ok=True, plots=True)
    model = YOLO(str(Path(trainer_model.trainer.save_dir) / 'weights/best.pt'))
else:
    print('Using the saved 30-epoch checkpoint; no retraining.')


## Full validation metrics
Use newly computed scores for the current checkpoint. Earlier reported metrics differ from the latest backup.

In [ ]:
metrics = model.val(data=str(CFG_PATH), imgsz=640, batch=16,
                    conf=0.001, iou=0.70, device=DEVICE,
                    project=str(RESULTS), name='validation_current',
                    exist_ok=True, plots=True)
row = {'Precision':float(metrics.box.mp), 'Recall':float(metrics.box.mr),
       'mAP50':float(metrics.box.map50), 'mAP50-95':float(metrics.box.map)}
frame = pd.DataFrame([row]); display(frame)
frame.to_csv(RESULTS / 'validation_metrics_current.csv', index=False)


## Ten validation prediction examples

In [ ]:
valid_images = sorted(p for p in (DATASET_ROOT/'valid/images').iterdir()
                      if p.suffix.lower() in {'.jpg','.jpeg','.png','.webp'})
EVIDENCE = RESULTS/'evidence'; VALID_OUT = EVIDENCE/'validation'
VALID_OUT.mkdir(parents=True, exist_ok=True)
for index, path in enumerate(random.Random(42).sample(valid_images,10),1):
    result = model.predict(str(path), imgsz=640, conf=0.25, iou=0.70,
                           device=DEVICE, verbose=False)[0]
    result.save(filename=str(VALID_OUT/f'validation_{index:02d}.jpg'))
print('Saved 10 validation predictions.')


## Five new-image predictions
Author-declared photographs. All outcomes are saved. This is qualitative evidence, not a labelled independent test set.

In [ ]:
NEW_IMAGES_DIR = Path('/content/new_images'); NEW_IMAGES_DIR.mkdir(exist_ok=True)
NEW_OUT = EVIDENCE/'new_images'; NEW_OUT.mkdir(parents=True, exist_ok=True)
summary = []
for i in range(1,6):
    name = f'new_{i:02d}.png'; path = NEW_IMAGES_DIR/name
    urllib.request.urlretrieve(
        f'https://raw.githubusercontent.com/aalbassiony-png/concrete-crack-detection/main/new_images/{name}', path)
    result = model.predict(str(path), imgsz=640, conf=0.25, iou=0.70,
                           device=DEVICE, verbose=False)[0]
    result.save(filename=str(NEW_OUT/name))
    summary.append({'image':name,'detections':len(result.boxes),
                    'confidences':result.boxes.conf.cpu().tolist()})
    display(Image(filename=str(NEW_OUT/name)))
display(pd.DataFrame(summary))
pd.DataFrame(summary).to_csv(NEW_OUT/'prediction_summary.csv',index=False)


## Current error candidates
Confidence 0.25; NMS IoU 0.70; greedy one-to-one matching IoU 0.50. Visually review candidates before labelling confirmed errors. Three FP and three FN remain to be documented for this checkpoint.

In [ ]:
all_valid_predictions = model.predict(source=[str(p) for p in valid_images],
    imgsz=640, conf=0.25, iou=0.70, device=DEVICE, verbose=False)


In [ ]:
import pandas as pd

MATCH_IOU = 0.50

def box_iou(a, b):
    x1 = max(a[0], b[0])
    y1 = max(a[1], b[1])
    x2 = min(a[2], b[2])
    y2 = min(a[3], b[3])

    intersection = max(0, x2 - x1) * max(0, y2 - y1)
    area_a = max(0, a[2] - a[0]) * max(0, a[3] - a[1])
    area_b = max(0, b[2] - b[0]) * max(0, b[3] - b[1])

    return intersection / max(area_a + area_b - intersection, 1e-9)

error_candidates = []
error_details = {}
total_tp = 0

for result in all_valid_predictions:
    image_path = Path(result.path)
    height, width = result.orig_shape

    label_path = (
        DATASET_ROOT / "valid" / "labels"
        / f"{image_path.stem}.txt"
    )
    assert label_path.exists(), f"Missing label file: {label_path}"

    gt_boxes = []
    for line in label_path.read_text().splitlines():
        if not line.strip():
            continue

        values = list(map(float, line.split()))
        assert len(values) == 5, f"Expected box annotation: {label_path}"

        class_id, cx, cy, bw, bh = values
        assert class_id == 0, "This analysis expects the cracks class only."

        gt_boxes.append([
            (cx - bw / 2) * width,
            (cy - bh / 2) * height,
            (cx + bw / 2) * width,
            (cy + bh / 2) * height
        ])

    pred_boxes = result.boxes.xyxy.cpu().tolist()
    confidences = result.boxes.conf.cpu().tolist()

    matched_gt = set()
    matched_pred = set()

    # Match predictions in descending confidence order.
    for pred_index in sorted(
        range(len(pred_boxes)),
        key=lambda i: confidences[i],
        reverse=True
    ):
        available_gt = [
            i for i in range(len(gt_boxes))
            if i not in matched_gt
        ]

        if not available_gt:
            continue

        gt_index = max(
            available_gt,
            key=lambda i: box_iou(pred_boxes[pred_index], gt_boxes[i])
        )

        if box_iou(pred_boxes[pred_index], gt_boxes[gt_index]) >= MATCH_IOU:
            matched_pred.add(pred_index)
            matched_gt.add(gt_index)

    fp_indices = [
        i for i in range(len(pred_boxes)) if i not in matched_pred
    ]
    fn_indices = [
        i for i in range(len(gt_boxes)) if i not in matched_gt
    ]
    total_tp += len(matched_pred)

    error_details[image_path.name] = {
        "image_path": image_path,
        "gt_boxes": gt_boxes,
        "pred_boxes": pred_boxes,
        "confidences": confidences,
        "fp_indices": fp_indices,
        "fn_indices": fn_indices
    }

    for i in fp_indices:
        error_candidates.append({
            "Image": image_path.name,
            "Error": "FP candidate",
            "Box index": i,
            "Confidence": confidences[i]
        })

    for i in fn_indices:
        error_candidates.append({
            "Image": image_path.name,
            "Error": "FN candidate",
            "Box index": i,
            "Confidence": None
        })

error_table = pd.DataFrame(
    error_candidates,
    columns=["Image", "Error", "Box index", "Confidence"]
)

ERROR_DIR = Path("/content/results/evidence/errors")
ERROR_DIR.mkdir(parents=True, exist_ok=True)
error_table.to_csv(ERROR_DIR / "error_candidates.csv", index=False)

display(error_table.round(3))

print("Matched detections:", total_tp)
print("FP candidates:", int((error_table["Error"] == "FP candidate").sum()))
print("FN candidates:", int((error_table["Error"] == "FN candidate").sum()))

## Execution record
The record documents this execution. Confirm fresh runtime, no secrets and no manual actions before recording a successful reproduction proof.

In [ ]:
proof = {'run_started_at_utc':STARTED_AT,
         'run_completed_at_utc':datetime.now(timezone.utc).isoformat(),
         'elapsed_minutes':(time.perf_counter()-START_TIME)/60,
         'python_version':__import__('platform').python_version(),
         'ultralytics_version':ultralytics.__version__, 'pytorch_version':torch.__version__,
         'device':torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU',
         'dataset_sha256':DATA_SHA256,'weights_sha256':WEIGHTS_SHA256,
         'full_training_requested':RUN_FULL_TRAINING,
         'active_weights_sha256':sha256_file(Path(model.ckpt_path)),
         'fresh_session_and_no_secrets_confirmed_by_author':False}
(RESULTS/'reproduction_run.json').write_text(json.dumps(proof,indent=2))
print('Run completed. Confirm the fresh-session procedure before claiming reproducibility proof.')
